# 03 — Feature Engineering and Preprocessing

**Objective:** Run deterministic domain features and demonstrate leakage-safe learned preprocessing on training data only.

In [1]:
from pathlib import Path
import os, sys
import pandas as pd
import plotly.express as px

_cwd = Path.cwd().resolve()
ROOT = _cwd.parent if _cwd.name == "dev" else _cwd
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("MPLCONFIGDIR", "/tmp/credit-risk-lab-matplotlib")

from credit_risk_lab.config.settings import settings
print(f"Project root: {settings.project_root}")

Project root: /Users/surelmanda/Mlops-Databricks-Projects/credit-risk-lab


## 1. Load the 90% training partition

In [2]:
from credit_risk_lab.infrastructure.data_sources import CSVDataSourceConfig, CSVDatasetRepository

train_raw = CSVDatasetRepository(CSVDataSourceConfig(path=settings.train_path)).load()
train_raw.shape

2026-07-11 09:52:05 | INFO     | CSVDatasetRepository | credit_risk_lab.infrastructure.data_sources.csv_dataset_repository:load:51 - Chargement du fichier : /Users/surelmanda/Mlops-Databricks-Projects/credit-risk-lab/data/processed/train.csv


2026-07-11 09:52:05 | INFO     | CSVDatasetRepository | credit_risk_lab.infrastructure.data_sources.csv_dataset_repository:load:59 - Dataset chargé (40493 lignes, 14 colonnes)


(40493, 14)

## 2. Deterministic feature engineering

These transformations learn no dataset statistics and are reused by the API.

In [3]:
from credit_risk_lab.infrastructure.feature_engineering import LoanFeatureEngineer

engineer = LoanFeatureEngineer()
train_features = engineer.transform(train_raw)
new_features = [c for c in train_features if c not in train_raw]
pd.DataFrame({"feature": new_features, "missing": train_features[new_features].isna().sum().values})

2026-07-11 09:52:05 | INFO     | loan_feature_engineer | credit_risk_lab.infrastructure.feature_engineering.pandas_feature_engineer:_log_shape:126 - [FeatureEngineering] Entrée - shape = (40493, 14)


2026-07-11 09:52:05 | INFO     | loan_feature_engineer | credit_risk_lab.infrastructure.feature_engineering.pandas_feature_engineer:_log_shape:126 - [FeatureEngineering] Sortie - shape = (40493, 39)


,feature,missing
0,dti,0
1,log_income,0
2,estimated_monthly_interest,0
3,interest_to_income,0
4,income_per_experience_year,0
5,credit_score_band,0
6,rate_per_score_point,0
7,age_group,0
8,exp_to_age,0
9,amnt_int_ratio,0


## 3. Leakage-safe preprocessing

The preprocessor is fitted only on a training subset and then reused unchanged.

In [4]:
from credit_risk_lab.application import three_way_stratified_split
from credit_risk_lab.infrastructure.modeling import build_preprocessor

split = three_way_stratified_split(train_features)
sensitive = [c for c in settings.sensitive_columns if c in split.x_train]
x_train = split.x_train.drop(columns=sensitive)
x_validation = split.x_validation.drop(columns=sensitive)
preprocessor = build_preprocessor(x_train)
train_matrix = preprocessor.fit_transform(x_train)
validation_matrix = preprocessor.transform(x_validation)
{"train_matrix": train_matrix.shape, "validation_matrix": validation_matrix.shape,
 "output_features": preprocessor.get_feature_names_out()[:20].tolist()}

{'train_matrix': (24295, 59),
 'validation_matrix': (8099, 59),
 'output_features': ['person_age',
  'person_income',
  'person_emp_exp',
  'loan_amnt',
  'loan_int_rate',
  'loan_percent_income',
  'cb_person_cred_hist_length',
  'credit_score',
  'dti',
  'log_income',
  'estimated_monthly_interest',
  'interest_to_income',
  'income_per_experience_year',
  'rate_per_score_point',
  'exp_to_age',
  'amnt_int_ratio',
  'loan_risk_score',
  'loan_intent_risk',
  'credit_hist_to_age',
  'age_first_credit']}

## Conclusion

The API repeats deterministic feature engineering but never refits the persisted learned preprocessor.